# Geometric encoding of turbulence for end-to-end quantum simulation

This notebook is the concise entry point for the modular workflow.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np

from geoencode_turb.config import WorkflowConfig

from geoencode_turb.quantum_encoding import (
    run_quantum_encoding,
    run_dense_quantum_encoding,
    run_sparse_quantum_encoding,
    benchmark_dense_vs_sparse,
    benchmark_dense_sparse_range,
)

from geoencode_turb.flow_reconstruction import reconstruct_flow_field

from geoencode_turb.flow_statistics import (
    compute_turbulence_scales,
    export_energy_spectrum,
    compute_energy_flux,
    export_spin_spectra,
    export_density_pdf,
    compute_vorticity_fields,
    export_velocity_vorticity_pdfs,
    compute_rq_statistics,
    run_structure_functions,
    report_anisotropy,
)

from geoencode_turb.visualization import (
    plot_encoded_amplitudes,
    plot_probability_histogram,
    render_vortex_isosurface,
    render_spin_isosurface,
    render_spinor_spectral_volume,
    render_density_spectral_volume,
    render_velocity_spectral_volume,
    render_vorticity_volume,
    render_density_volume,
)

from geoencode_turb.affine_fit_analysis import (
    run_conditional_angle_analysis,
    plot_affine_fit_error_comparison,
)

from geoencode_turb.temporal_correlation import (
    compute_two_time_velocity_correlation,
    estimate_spinor_dephasing_time,
    evolve_flow_field,
)

config = WorkflowConfig()
config.ensure_output_directories()

## 0. Configuration

In [4]:
from importlib import reload
import geoencode_turb.config as config_module

reload(config_module)
config = config_module.WorkflowConfig()
config.ensure_output_directories()
config

WorkflowConfig(nx=10, ny=10, nz=10, k_cutoff=15.0, seed_spin_up=2025, seed_spin_down=2026, max_parallel_threads=16, velocity_method='FDM', derivative_method='FDM', fontsize=8.0, data_dir=WindowsPath('data/3D'), figure_dir=WindowsPath('figures'))

## 1. Quantum state preparation

In [33]:
# encoding = run_dense_quantum_encoding(config)

encoding = run_sparse_quantum_encoding(config, tail_tolerance=1.0e-12, locality_radius=3)

# amplitude_figure = plot_encoded_amplitudes(encoding, config)
# probability_figure = plot_probability_histogram(encoding, config)

Compiler: sparse
Compilation time: 0.710738 s
Significant modes: 30,047
Occupied prefix-tree nodes: 71,710
Estimated tail probability: 6.728058e-13
Simulating circuits...
   Simulation done in 2.5166 s
Transpiling...
Transpile done in 0.0583 s
Transpiled circuit depth: 235


### Optional dense-sparse compiler benchmark

Run this separately from the production workflow. Full-state and flow comparisons retain the classical $O(2^n)$ validation cost.

In [ ]:
benchmark = benchmark_dense_vs_sparse(
    config,
    tail_tolerance=1.0e-12,
    locality_radius=3,
    simulate_statevectors=True,
    compute_physical_diagnostics=True,
)

## 2. Pauli-spinor and flow-field reconstruction

In [34]:
flow = reconstruct_flow_field(encoding, config)

vorticity = compute_vorticity_fields(flow, config)

# scale_diagnostics = compute_turbulence_scales(
#     flow.ux,
#     flow.uy,
#     flow.uz,
#     L_box=2.0 * np.pi,
#     k_L=1.0,
#     k_eta_over_kmax=5.0,
#     C_epsilon=1.0,
#     nu=None,
#     Re_L=None,
#     component_axes=(0, 1, 2),
# )

## 3. Calculate turbulent statistics

In [19]:
energy_spectrum = export_energy_spectrum(flow, config)
# energy_flux = compute_energy_flux(flow, config)
# spin_fields, spin_spectra = export_spin_spectra(flow, config)
# density_pdf = export_density_pdf(flow, config)
# field_pdfs = export_velocity_vorticity_pdfs(flow, vorticity, config)
# anisotropy = report_anisotropy(flow)
# rq_statistics = compute_rq_statistics(flow, config)

# This retains the original 1,000,000,000 sampled pairs per order.
# structure_functions = run_structure_functions(flow, config)

## 4. Optional high-cost analyses

Run these calls individually when the corresponding output is needed.

In [ ]:
# PyVista renderings retained from the original notebook.
render_vortex_isosurface(
    flow,
    vorticity,
    iso_value=350,
    helicity_limit=4000,
    output_path=config.figure_dir / f"iso-vorticity_n={config.nx + config.ny + config.nz}.png",
)

# render_spin_isosurface(flow, spin_fields, vorticity)
# render_spinor_spectral_volume(flow)
# render_density_spectral_volume(flow)
# render_velocity_spectral_volume(flow)
# render_vorticity_volume(vorticity)
# render_density_volume(flow)

Vorticity isosurface saved successfully: figures\iso-vorticity_n=24.png


WindowsPath('figures/iso-vorticity_n=24.png')

Compute two time velocity correlation

In [ ]:
import gc

flow.psi1 = None
flow.psi2 = None
flow.rho = None
flow.uy = None
flow.uz = None
flow.KX = None
flow.KY = None
flow.KZ = None
flow.K2 = None
flow.ik2 = None

gc.collect()

omega_mean, delta_omega, tau_phi = (
    estimate_spinor_dephasing_time(flow)
)

print(f"Mean frequency     = {omega_mean:.6e}")
print(f"Frequency spread   = {delta_omega:.6e}")
print(f"Dephasing time     = {tau_phi:.6e}")

tau = np.linspace(0.0, 4.0 * tau_phi, 100)

temporal_correlation = compute_two_time_velocity_correlation(
    flow,
    tau,
    config,
    component="x",
    kinetic_coefficient=0.5,
    density_floor_relative=0.0,
    chunk_size=2_000_000,
    output_filename=f"two_time_velocity_correlation_n={config.nx + config.ny + config.nz}.csv",
)

Plot vortex structures at a given $t$.

In [ ]:
tau_snapshot = 3

flow_tau = evolve_flow_field(flow, tau_snapshot, config)
vorticity_tau = compute_vorticity_fields(flow_tau, config)

render_vortex_isosurface(
    flow_tau,
    vorticity_tau,
    iso_value=300,
    helicity_limit=3000,
    output_path=config.figure_dir / f"iso-vorticity_n={config.nx + config.ny + config.nz}_t={tau_snapshot:.4f}.png",
)

## 5. Conditional-angle affine-fit diagnostics

In [ ]:
affine_fit = run_conditional_angle_analysis(encoding, config)
affine_fit_figure = plot_affine_fit_error_comparison(affine_fit)